In [10]:
import logging

from pulao.logging import init_logging

import pandas as pd

from pulao.bar import SBarManager,SBar,CBarManager

import polars as pl
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from pulao.constant import Exchange, FractalType, Timeframe

from pulao.logging import get_logger
init_logging(level=logging.DEBUG)
open("./logs/pulao.log", "w").close() # 每次运行前清空日志
logger = get_logger(__name__)
logger.debug("开始运行")

df = pl.read_csv("../dataset/I8888.XDCE_60m.csv", try_parse_dates=True)
df = df.head(300)  # test

sbar_list = []
columns = df.columns

symbol="i8888"
exchange = Exchange.DCE
tieframe = Timeframe.H1
for idx, row in enumerate(df.iter_rows()):
    row_dict = dict(zip(columns, row, strict=False))
    # datetime,open,close,high,low,volume,money,open_interest,signal
    datetime = row_dict["datetime"]
    o = row_dict["open"]
    close = row_dict["close"]
    high = row_dict["high"]
    low = row_dict["low"]
    volume = row_dict["volume"]
    money = row_dict["money"]
    open_interest = row_dict["open_interest"]

    sbar = SBar(id=idx, symbol=symbol, exchange=exchange, timeframe=tieframe, datetime=datetime, turnover=money, open_price=o, close_price=close, high_price=high, low_price=low, volume=volume,open_interest=open_interest, ema_short=0, ema_long=0, created_at=datetime)

    sbar_list.append(sbar)
# 模拟行情数据接收
sbar_manager = SBarManager(symbol=symbol, timeframe=tieframe)
cbar_manager = CBarManager(sbar_manager=sbar_manager)

# 流入模拟数据
for sbar in sbar_list:
    sbar_manager.append(sbar)

logger.debug("运行结束")
# sleep(3) # 等一会，让日志输出完

#
# Plotly 绘图
#
# 设置全局默认布局
pio.templates.default = 'plotly_dark'
#
# region . Plotly - Sbar
#
df_sbar = sbar_manager.df_sbar.to_pandas()
df_sbar["datetime"] = pd.date_range("2025-01-01", periods=df_sbar.shape[0], freq="h")
# df_sbar["open_price"] = df_sbar["high_price"]
# df_sbar["close_price"] = df_sbar["low_price"]
df_sbar.insert(0, "index", range(len(df_sbar)))

fig = make_subplots(
    rows=1, cols=1
)
fig.add_trace(go.Candlestick(
    x=df_sbar['datetime'],
    open=df_sbar['open_price'],
    high=df_sbar['high_price'],
    low=df_sbar['low_price'],
    close=df_sbar['close_price'],
    name='K线',
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_sbar['datetime'],
    y=df_sbar['close_price'] * 0.995,   # 放在K线最低价下方一点，不遮挡蜡烛
    mode='text',
    text=df_sbar["index"],      # ← 添加序号
    textposition="bottom center",              # ← 文字位置
    textfont=dict(size=10, color="white"),
), row=1, col=1)

title = 'SBar Chart - 原始K线数据'
fig.update_layout(title=title,
                  height=900,
                  hovermode='x unified',  # X轴悬停联动虚线
                  )

fig.update_xaxes(
    showgrid=False,
)

fig.update_yaxes(
    showgrid=False,
)

fig.show()
# endregion

#
# region . Plotly - Cbar
#
df_cbar = cbar_manager.df_cbar.to_pandas()
df_cbar["datetime"] = pd.date_range("2025-01-01", periods=df_cbar.shape[0], freq="h")
df_cbar["open_price"] = df_cbar["high_price"]
df_cbar["close_price"] = df_cbar["low_price"]
df_cbar.insert(0, "index", range(len(df_cbar)))

fig = make_subplots(
    rows=1, cols=1
)
fig.add_trace(go.Candlestick(
    x=df_cbar['datetime'],
    open=df_cbar['open_price'],
    high=df_cbar['high_price'],
    low=df_cbar['low_price'],
    close=df_cbar['close_price'],
    name='K线',
), row=1, col=1)

df_fractal_bottom = df_cbar[(df_cbar['fractal_type'] == FractalType.BOTTOM)]

fig.add_trace(go.Scatter(
    x=df_fractal_bottom['datetime'],
    y=df_fractal_bottom['low_price'] * 0.995,   # 放在K线最低价下方一点，不遮挡蜡烛
    mode='markers+text',
    name='swing point low',
    marker=dict(
        symbol='triangle-up',
        size=14,
        color='#1E90FF',
    ),
    text=df_fractal_bottom["index"],      # ← 添加序号
    textposition="bottom center",              # ← 文字位置
    textfont=dict(size=10, color="white"),
    hovertemplate=
        "<b>波段低点</b><br>" +
        "时间: %{x}<br>" +
        "价格: %{y:.2f}<br>" +
        "<extra></extra>"
), row=1, col=1)


df_fractal_top = df_cbar[(df_cbar['fractal_type'] == FractalType.TOP)]

fig.add_trace(go.Scatter(
    x=df_fractal_top['datetime'],
    y=df_fractal_top['high_price'] * 1.005,  # 放在K线最高价上方一点
    mode='markers+text',
    name='swing point high',
    marker=dict(
        symbol='triangle-down',
        size=14,
        color='#FF4500',
    ),
    text=df_fractal_top["index"],      # ← 添加序号
    textposition="top center",              # ← 文字位置
    textfont=dict(size=10, color="white"),
    hovertemplate=
        "<b>波段高点</b><br>" +
        "时间: %{x}<br>" +
        "价格: %{y:.2f}<br>" +
        "<extra></extra>"
), row=1, col=1)

title = 'CBar Chart - K线包含处理-分形标记'
fig.update_layout(title=title,
                  height=900,
                  hovermode='x unified',  # X轴悬停联动虚线
                  )

fig.update_xaxes(
    showgrid=False,
)

fig.update_yaxes(
    showgrid=False,
)

fig.show()
# endregion



{"event": "开始运行", "level": "debug", "logger": "__main__", "time": "2026-03-25 16:03:03"}


{"event": "运行结束", "level": "debug", "logger": "__main__", "time": "2026-03-25 16:03:04"}


'6.6.0'